In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
import pandas as pd
import seaborn as sns
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import confusion_matrix, classification_report
from PIL import Image

In [ ]:
import tensorflow as tf
from keras import preprocessing
from keras.models import load_model
from keras.src.legacy.preprocessing.image import ImageDataGenerator
from keras import layers
from keras.models import Sequential
from keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from keras.applications import MobileNetV2

## Data Prep

In [ ]:
# local imports
from func import preprocess_data, keras_ds_train_test_split, clean_rawimg, download_dataset

# Dataset
df, data_dir = download_dataset(path=r'C:\Users\paudu\Desktop\coding\classes\homework\February\Project')

In [ ]:
translate = {"ragno":"spider" ,"cane": "dog", "cavallo": "horse", "elefante": "elephant", "farfalla": "butterfly", "gallina": "chicken", "gatto": "cat", "mucca": "cow", "pecora": "sheep", "scoiattolo": "squirrel", "dog": "cane", "cavallo": "horse", "elephant" : "elefante", "butterfly": "farfalla", "chicken": "gallina", "cat": "gatto", "cow": "mucca", "spider": "ragno", "squirrel": "scoiattolo"}

filepaths = []
labels = []

for label in os.listdir(data_dir):
    class_path = os.path.join(data_dir, label)
    if os.path.isdir(class_path):
        for file in os.listdir(class_path):
            filepaths.append(os.path.join(class_path, file))
            labels.append(translate[label])

df = pd.DataFrame({
    "filepath": filepaths,
    "label": labels
})

df

## Data Loading

In [ ]:
figure, axes = plt.subplots(2, 5, figsize=(18, 6))
figure.suptitle("Sample Images from Each Class", fontsize=12)

images = []
for class_name in sorted(os.listdir(data_dir)):
    class_path = os.path.join(data_dir, class_name)
    if not os.path.isdir(class_path):
        continue
    class_images = [f for f in os.listdir(class_path) if os.path.isfile(os.path.join(class_path, f))]
    if not class_images:
        continue
    images.append(os.path.join(class_path, class_images[np.random.randint(0, len(class_images))])) 

for i, img_path in enumerate(images[:20]):
    img = Image.open(img_path)
    row, col = divmod(i, 5)
    axes[row, col].imshow(img)
    axes[row, col].set_title(
        f"{translate[os.path.basename(os.path.dirname(img_path))]}"
    )
    axes[row, col].axis('off')

## Data preprocessing

In [ ]:
from sklearn.model_selection import train_test_split

# 70% train
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df["label"],
    random_state=42
)

# 15% val, 15% test
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=42
)

print("Train:", len(train_df))
print("Val:", len(val_df))
print("Test:", len(test_df))

In [ ]:
counts = train_df["label"].value_counts().values
names = train_df["label"].value_counts().index

# Color gradient
colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(names)))

plt.figure(figsize=(12, 6))
bars = plt.bar(names, counts, color=colors, edgecolor='black', linewidth=0.5)

# Add count labels on bars
for bar, count in zip(bars, counts):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20, 
             str(count), ha='center', fontsize=10, fontweight='bold')

plt.xlabel('Class', fontsize=12)
plt.ylabel('Number of Images', fontsize=12)
plt.title('Training Data: Class Distribution', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.axhline(y=np.mean(counts), color='red', linestyle='--', linewidth=2, label=f'Mean: {np.mean(counts):.0f}')
plt.legend()
plt.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nTotal training images: {sum(counts)}")
print(f"Min: {names[-1]} ({counts[-1]}), Max: {names[0]} ({counts[0]})")
print(f"Imbalance ratio: {max(counts)/min(counts):.2f}x")

## split data

In [ ]:
INPUT_SHAPE = (224,224)
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    shear_range=0.2,
    zoom_range=0.2,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest'
)

test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_dataframe(
    train_df,
    x_col="filepath",
    y_col="label",
    target_size=INPUT_SHAPE,
    batch_size=BATCH_SIZE,
    class_mode="categorical"
)

val_generator = test_datagen.flow_from_dataframe(
    val_df,
    x_col="filepath",
    y_col="label",
    target_size=INPUT_SHAPE,
    batch_size=BATCH_SIZE,
    class_mode="categorical"
)

test_generator = test_datagen.flow_from_dataframe(
    test_df,
    x_col="filepath",
    y_col="label",
    target_size=INPUT_SHAPE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

## Modeling

In [ ]:
# checkpoint saves the best model based on validation accuracy during training
checkpoint = ModelCheckpoint(
    filepath='/kaggle/working/model-checkpoint/best_model_stage1.keras',   
    monitor='val_accuracy',
    save_best_only=True,
    mode='max',
)

# early stopping stop training if val_loss does not improve for 5 epochs
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',    
    factor=0.2,            
    patience=3,            
    verbose=1,             
    min_lr=1e-7            
)

In [ ]:
classes = np.unique(train_generator.classes)
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=train_generator.classes
)

class_weights_dict = dict(enumerate(class_weights))
class_weights_dict

In [ ]:
base_model = MobileNetV2(weights='imagenet', 
                         include_top=False, 
                         input_shape=(224, 224, 3))
base_model.trainable = True
for layer in base_model.layers[:-20]: 
    layer.trainable = False

model = Sequential([
    base_model,
    layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(128, (3, 3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.5),
    layers.Dense(10, activation='softmax')
])

model.compile(optimizer='adam', 
              loss='categorical_crossentropy', 
              metrics=['accuracy'])
model.summary()

## Training

In [ ]:
if os.path.exists('./models/model.keras'):
    check = input("Model already exists. Would you like to retrain? (y/n): ")

if check.lower() == 'y':
    model.fit(
        train_generator,
        validation_data=val_generator,
        epochs=30,
        class_weight=class_weights_dict,
        callbacks=[checkpoint, early_stop, reduce_lr]
    )
    model.save('/models/model.keras')
    np.save('./histories/model-history', model.history.history)

In [ ]:
model.load_weights('/kaggle/working/model-checkpoint/best_model_stage1.keras')
model.history.history = np.load('./histories/model-history.npy', allow_pickle=True).item()

## Evaluation and Visualization

In [ ]:
loss, accuracy = model.evaluate(train_generator)
print("Evaluation on data train: ")
print(f"accuracy: {accuracy*100:.2f}")
print(f"loss: {loss}")

In [ ]:
loss, accuracy = model.evaluate(test_generator)
print("Evaluation on data test: ")
print(f"accuracy: {accuracy*100:.2f}")
print(f"loss: {loss}")

In [ ]:
loss, accuracy = model.evaluate(val_generator)
print("Evaluation on data validation: ")
print(f"accuracy: {accuracy*100:.2f}")
print(f"loss: {loss}")

In [ ]:
# Prediksi data test
Y_pred = model.predict(test_generator)
y_pred = np.argmax(Y_pred, axis=1)

# Ambil label asli
y_true = test_generator.classes
class_names = list(test_generator.class_indices.keys())

# Print Report
print(classification_report(y_true, y_pred, target_names=class_names))

# Plot Heatmap
plt.figure(figsize=(10, 8))
cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', 
            xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix: Animals-10 Edition')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

In [ ]:
acc = model.history.history['accuracy']
val_acc = model.history.history['val_accuracy']
loss = model.history.history['loss']
val_loss = model.history.history['val_loss']

epochs_range = range(len(acc))

plt.figure(figsize=(15, 5))

# Plot Accuracy
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Training Accuracy', color='#1DB954', lw=2)
plt.plot(epochs_range, val_acc, label='Validation Accuracy', color='#191414', lw=2)
plt.title('Accuracy Curves', fontsize=14)
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend(loc='lower right')
plt.grid(alpha=0.3)

# Plot Loss
plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Training Loss', color='#1DB954', lw=2)
plt.plot(epochs_range, val_loss, label='Validation Loss', color='#191414', lw=2)
plt.title('Loss Curves', fontsize=14)
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend(loc='upper right')
plt.grid(alpha=0.3)

plt.show()